# ApexPlanet Internship — Task 1: Data Immersion & Wrangling

Goal: inspect, clean, validate and export the UCI Online Retail dataset. Run all cells from top to bottom.

**Rules:** remove exact duplicates; strip column-name spaces; fill missing Description with `Unknown`; retain missing CustomerID; parse dates and numeric fields; calculate rounded Sales; classify transaction types with negative UnitPrice taking precedence.

## 1. Upload the dataset

In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import files

uploaded = files.upload()
if not uploaded: raise ValueError("Please upload the Online Retail Excel or CSV file.")
file_name = next(iter(uploaded))
if file_name.lower().endswith((".xlsx", ".xls")): df = pd.read_excel(file_name)
elif file_name.lower().endswith(".csv"): df = pd.read_csv(file_name)
else: raise ValueError("Upload .xlsx, .xls, or .csv")
df.columns = df.columns.astype(str).str.strip()
print("File:", file_name, "| Shape:", df.shape)
display(df.head())

## 2. Initial data quality assessment

In [ ]:
expected = ["InvoiceNo", "StockCode", "Description", "Quantity", "InvoiceDate", "UnitPrice", "CustomerID", "Country"]
missing_columns = [c for c in expected if c not in df.columns]
if missing_columns: raise ValueError(f"Required columns missing: {missing_columns}")
rows_before = len(df)
duplicates_removed = int(df.duplicated().sum())
print("Rows:", len(df), "| Columns:", len(df.columns), "| Exact duplicates:", duplicates_removed)
display(df.describe(include="all").T)
display(pd.DataFrame({"Missing values":df.isna().sum(),"Missing %":(df.isna().mean()*100).round(2),"Data type":df.dtypes.astype(str)}))

## 3. Clean and transform

In [ ]:
df_clean = df.drop_duplicates().copy()
df_clean.columns = df_clean.columns.astype(str).str.strip()
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"], errors="coerce")
for c in ["Quantity", "UnitPrice", "CustomerID"]: df_clean[c] = pd.to_numeric(df_clean[c], errors="coerce")
df_clean["Description"] = df_clean["Description"].fillna("Unknown")
# Keep missing CustomerID values; do not invent IDs.
df_clean["Sales"] = (df_clean["Quantity"] * df_clean["UnitPrice"]).round(2)
# Negative UnitPrice takes precedence over negative Quantity.
df_clean["TransactionType"] = "Normal"
df_clean.loc[df_clean["UnitPrice"].eq(0), "TransactionType"] = "Zero Price"
df_clean.loc[df_clean["Quantity"].lt(0), "TransactionType"] = "Return"
df_clean.loc[df_clean["UnitPrice"].lt(0), "TransactionType"] = "Price Adjustment"
print(f"Rows before: {rows_before:,}; duplicates removed: {duplicates_removed:,}; rows after: {len(df_clean):,}")
display(df_clean.head())

## 4. Validate the cleaned data

In [ ]:
quality_after = pd.DataFrame({"Missing values":df_clean.isna().sum(),"Missing %":(df_clean.isna().mean()*100).round(2),"Data type":df_clean.dtypes.astype(str)})
display(quality_after)
print("Remaining duplicates:", int(df_clean.duplicated().sum()))
print("Sales formula mismatches:", int((df_clean["Sales"] != (df_clean["Quantity"]*df_clean["UnitPrice"]).round(2)).fillna(False).sum()))
print("Negative Quantity:", int(df_clean["Quantity"].lt(0).sum()))
print("Zero UnitPrice:", int(df_clean["UnitPrice"].eq(0).sum()))
print("Negative UnitPrice:", int(df_clean["UnitPrice"].lt(0).sum()))
print("Missing CustomerID retained:", int(df_clean["CustomerID"].isna().sum()))
display(df_clean["TransactionType"].value_counts().rename_axis("TransactionType").to_frame("Rows"))

## 5. Create the data dictionary

In [ ]:
meanings = {
"InvoiceNo":("Invoice number; values beginning with C commonly indicate cancelled transactions.","Identify invoices and investigate cancellations or returns."),
"StockCode":("Code identifying the product.","Group transactions by product."),
"Description":("Product description; missing values are represented as Unknown.","Understand which products were purchased."),
"Quantity":("Number of units; negative values can indicate returns.","Measure units sold or returned."),
"InvoiceDate":("Date and time of the transaction.","Analyze sales patterns over time."),
"UnitPrice":("Price per unit; the UCI dataset uses GBP.","Analyze pricing and calculate transaction value."),
"CustomerID":("Customer identifier; some transactions have no customer ID.","Analyze customer behavior when an ID is available."),
"Country":("Country associated with the transaction.","Compare transaction activity across countries."),
"Sales":("Quantity multiplied by UnitPrice, rounded to two decimals.","Estimate transaction line value; negative values may represent returns or adjustments."),
"TransactionType":("Rule-based label; negative UnitPrice takes precedence.","Separate unusual transaction types for analysis.")}
lines=["# Data Dictionary — UCI Online Retail","","## Dataset overview","",f"- Rows before duplicate removal: {rows_before:,}",f"- Exact duplicate rows removed: {duplicates_removed:,}",f"- Rows in cleaned output: {len(df_clean):,}",f"- Columns in cleaned output: {len(df_clean.columns)}","","## Data quality notes","",f"- Remaining exact duplicates: {int(df_clean.duplicated().sum()):,}",f"- Missing Description after handling: {int(df_clean['Description'].isna().sum()):,}",f"- Missing CustomerID retained: {int(df_clean['CustomerID'].isna().sum()):,}",f"- Negative Quantity rows: {int(df_clean['Quantity'].lt(0).sum()):,}",f"- Zero UnitPrice rows: {int(df_clean['UnitPrice'].eq(0).sum()):,}",f"- Negative UnitPrice rows: {int(df_clean['UnitPrice'].lt(0).sum()):,}","- Description missing values are filled with Unknown; missing CustomerID values are retained.","- Sales = Quantity × UnitPrice, rounded to two decimals.","- TransactionType precedence: negative UnitPrice → Price Adjustment; otherwise negative Quantity → Return; otherwise zero UnitPrice → Zero Price; otherwise Normal.","","## Column definitions","","| Column | Data type | Meaning | Business relevance |","|---|---|---|---|"]
for col in df_clean.columns:
 meaning,relevance=meanings[col]; vals=[str(col),str(df_clean[col].dtype),meaning,relevance]; lines.append("| "+" | ".join(v.replace("|",r"\|") for v in vals)+" |")
lines += ["","## Transaction type counts","","| Transaction type | Rows |","|---|---:|"]
for label,count in df_clean["TransactionType"].value_counts().items(): lines.append(f"| {label} | {int(count):,} |")
Path("data_dictionary.md").write_text("\n".join(lines)+"\n",encoding="utf-8")
print("Created data_dictionary.md")

## 6. Export deliverables

In [ ]:
csv_path = "final_cleaned_online_retail.csv"
df_clean.to_csv(csv_path, index=False, date_format="%Y-%m-%d %H:%M:%S", float_format="%.2f")
print("Created:", csv_path, "and data_dictionary.md")
files.download(csv_path)
files.download("data_dictionary.md")

## 7. Conclusion

The workflow profiles the data, removes exact duplicates, standardizes column names and data types, fills missing descriptions, retains missing customer IDs, calculates Sales, labels transaction types, validates the output, and exports the cleaned CSV and data dictionary. Run all cells before submitting. GitHub requires the notebook, Python script, cleaned dataset and dictionary; record the 3–5 minute LinkedIn walkthrough separately.